Enrollment No: 240840116029

Practical 9 : Design and develop a simple interactive dashboard using Dash/Plotly for business reporting.

Step 1: Install required libraries

In [ ]:
# Install required libraries
!pip install dash plotly pandas

Step 2: Import libraries

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
from google.colab import output
import threading

Step 3: Create dataset

In [ ]:
# Create sample e-commerce sales dataset
data = {
    "Month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],

    "Category": ["Electronics", "Clothing", "Furniture",
                 "Electronics", "Clothing", "Furniture",
                 "Electronics", "Clothing", "Furniture",
                 "Electronics", "Clothing", "Furniture"],

    "Payment": ["UPI", "Card", "Cash", "Card",
                "UPI", "Cash", "UPI", "Card",
                "Cash", "Card", "UPI", "Cash"],

    "Sales": [25000, 18000, 12000, 30000, 22000, 15000,
              35000, 28000, 20000, 40000, 32000, 25000],

    "Orders": [25, 18, 12, 30, 22, 15,
               35, 28, 20, 40, 32, 25]
}

# Convert data into DataFrame
df = pd.DataFrame(data)

# Display dataset
df


,Month,Category,Payment,Sales,Orders
0,Jan,Electronics,UPI,25000,25
1,Feb,Clothing,Card,18000,18
2,Mar,Furniture,Cash,12000,12
3,Apr,Electronics,Card,30000,30
4,May,Clothing,UPI,22000,22
5,Jun,Furniture,Cash,15000,15
6,Jul,Electronics,UPI,35000,35
7,Aug,Clothing,Card,28000,28
8,Sep,Furniture,Cash,20000,20
9,Oct,Electronics,Card,40000,40


Step 4: Update dropdown filters

In [ ]:

# Create Dash application
app = Dash(__name__)

# Define dashboard layout
app.layout = html.Div([

    html.H1("E-Commerce Sales Dashboard",
            style={"textAlign": "center",
                   "color": "#1A237E"}),

    html.Hr(),

    # Category filter
    html.Label("Select Category:"),

    dcc.Dropdown(
        id="category",
        options=[{"label": "All", "value": "All"}] +
                [{"label": c, "value": c}
                 for c in df["Category"].unique()],
        value="All",
        clearable=False
    ),

    html.Br(),

    # Payment filter
    html.Label("Select Payment Method:"),

    dcc.Dropdown(
        id="payment",
        options=[{"label": "All", "value": "All"}] +
                [{"label": p, "value": p}
                 for p in df["Payment"].unique()],
        value="All",
        clearable=False
    ),

    html.Br(),

    # Dashboard summary and charts
    html.Div(id="sales_summary"),
    dcc.Graph(id="sales_chart"),
    dcc.Graph(id="product_chart"),
    dcc.Graph(id="region_chart")

], style={
    "backgroundColor": "#F4F6F9",
    "padding": "25px",
    "fontFamily": "Arial"
})


Step 5: Interactive dashboard functionality

In [ ]:

# Define callback function
@app.callback(
    [
        Output("sales_summary", "children"),
        Output("sales_chart", "figure"),
        Output("product_chart", "figure"),
        Output("region_chart", "figure")
    ],
    [
        Input("category", "value"),
        Input("payment", "value")
    ]
)

# Update dashboard according to filters
def update_dashboard(category, payment):

    # Copy original dataset
    filtered = df.copy()

    # Apply category filter
    if category != "All":
        filtered = filtered[filtered["Category"] == category]

    # Apply payment filter
    if payment != "All":
        filtered = filtered[filtered["Payment"] == payment]

    # Calculate total sales, orders and average sales
    total_sales = filtered["Sales"].sum()
    total_orders = filtered["Orders"].sum()
    avg_sales = filtered["Sales"].mean()

    # Create summary cards
    summary = html.Div([

        html.Div([
            html.H4("Total Revenue"),
            html.H2(f"₹{total_sales:,}")
        ]),

        html.Div([
            html.H4("Total Orders"),
            html.H2(f"{total_orders}")
        ]),

        html.Div([
            html.H4("Average Sales"),
            html.H2(f"₹{avg_sales:,.2f}")
        ])

    ], style={
        "display": "flex",
        "justifyContent": "space-around",
        "backgroundColor": "#DCEBFF",
        "padding": "15px"
    })

    # Monthly sales line chart
    fig1 = px.line(
        filtered,
        x="Month",
        y="Sales",
        markers=True,
        title="Monthly Revenue Analysis"
    )

    # Category-wise sales bar chart
    category_data = filtered.groupby(
        "Category", as_index=False
    )["Sales"].sum()

    fig2 = px.bar(
        category_data,
        x="Category",
        y="Sales",
        title="Category-wise Sales"
    )

    # Payment method pie chart
    payment_data = filtered.groupby(
        "Payment", as_index=False
    )["Sales"].sum()

    fig3 = px.pie(
        payment_data,
        names="Payment",
        values="Sales",
        title="Sales by Payment Method"
    )

    # Return all dashboard components
    return summary, fig1, fig2, fig3


Step 6: Run dashboard in Google Colab

In [ ]:
# Define function to start Dash server
def run_app():
    app.run(
        host="0.0.0.0",
        port=8050,
        debug=False
    )

# Run dashboard in background thread
thread = threading.Thread(target=run_app)
thread.start()

# Display dashboard inside Colab
output.serve_kernel_port_as_iframe(
    8050,
    height=900
)